# rtp-audit pilot (Colab)

Before running: add `HF_TOKEN` in Colab **Secrets** (needed for gated Llama repos; accept the licence on the Hub first).
Cells mirror spec section 13. Every stage resumes from finished shards, so after a disconnect just re-run the cell.

In [ ]:
# Cell 1: mount Drive and set environment variables
from google.colab import drive, userdata
import os

drive.mount("/content/drive")

os.environ["RTP_WORK_DIR"] = "/content/drive/MyDrive/rtp"
os.environ["HF_HOME"] = "/content/hf"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.makedirs(os.environ["RTP_WORK_DIR"], exist_ok=True)

In [ ]:
# Cell 2: get the code and install (torch is not reinstalled)
!nvidia-smi --query-gpu=name,memory.total --format=csv
REPO_URL = "<YOUR_REPO_URL>"  # replace before running
!test -d /content/rtp-audit || git clone {REPO_URL} /content/rtp-audit
%cd /content/rtp-audit
!git pull
!pip -q install -e ".[dev]"

In [ ]:
# Cell 3: smoke test (a few minutes)
!python -m rtp.cli smoke
!pytest -q

In [ ]:
# Cell 4: generate data (only once)
!python -m rtp.cli build-data --config configs/pilot.yaml

In [ ]:
# Cell 5: behavioural measurement (resumable: re-run this cell after an interruption)
MODELS = [
    "allenai/Llama-3.1-Tulu-3-8B-SFT",
    "allenai/Llama-3.1-Tulu-3-8B-DPO",
]
for m in MODELS:
    !python -m rtp.cli measure --config configs/pilot.yaml --model {m}
    !python -m rtp.cli open-ended --config configs/pilot.yaml --model {m}

In [ ]:
# Cell 6: view the summary
!python -m rtp.cli report --config configs/pilot.yaml